# HuggingFace Local Evaluation Quickstart

This cookbook shows how to instrument a small local HuggingFace-style pipeline with TruLens OTEL tracing and configure local HuggingFace feedback without external API calls.


## Install dependencies

```bash
pip install trulens trulens-providers-huggingface transformers torch
```


In [ ]:
from trulens.apps.app import TruApp
from trulens.core import Metric, Selector, TruSession
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes


## Define a tiny local pipeline

The example uses a deterministic in-memory pipeline so the instrumentation pattern is easy to see. Replace `generate` with a local `transformers.pipeline` call in a real app.


In [ ]:
class LocalQAPipeline:
    @instrument(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        attributes={
            SpanAttributes.RETRIEVAL.QUERY_TEXT: "question",
            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
        },
    )
    def retrieve(self, question: str) -> list[str]:
        return [
            "TruLens helps evaluate and track LLM application quality with feedback functions."
        ]

    @instrument(span_type=SpanAttributes.SpanType.GENERATION)
    def generate(self, question: str, context: list[str]) -> str:
        return context[0]

    @instrument(
        span_type=SpanAttributes.SpanType.RECORD_ROOT,
        attributes={
            SpanAttributes.RECORD_ROOT.INPUT: "question",
            SpanAttributes.RECORD_ROOT.OUTPUT: "return",
        },
    )
    def answer(self, question: str) -> str:
        context = self.retrieve(question)
        return self.generate(question, context)


## Configure local HuggingFace feedback

`HuggingfaceLocal` runs feedback locally. The exact model you choose depends on the feedback function and resources available on your machine.


In [ ]:
from trulens.providers.huggingface import HuggingfaceLocal

# Choose a small local model appropriate for your feedback task.
# This keeps the example API-only; update the model name for your environment.
provider = HuggingfaceLocal(model_engine="distilbert-base-uncased")

# Example custom metric hook: use selector wiring with the recorded input/output.
# Replace `provider.language_match` with the local feedback function you want to run.
f_language_match = Metric(
    implementation=provider.language_match,
    name="Language Match",
    selectors={
        "text1": Selector.select_record_input(),
        "text2": Selector.select_record_output(),
    },
)


## Record and evaluate


In [ ]:
session = TruSession()
session.reset_database()

app = LocalQAPipeline()
tru_app = TruApp(
    app,
    app_name="HuggingFace Local Quickstart",
    app_version="v1",
    feedbacks=[f_language_match],
)

with tru_app as recording:
    app.answer("What is TruLens used for?")

recording.retrieve_feedback_results(timeout=300)
session.get_leaderboard()


## View traces and scores

```python
from trulens.dashboard import run_dashboard
run_dashboard(session)
```

Use the dashboard to inspect the root span, retrieval span, generation span, and local HuggingFace feedback results.
